# Moments: free quality test

Puts a character into **your own clip** with the open-source **Wan VACE** video model, on Colab's free GPU.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells in order (▶ on each, or **Runtime → Run all** and answer the upload prompts).

What to film (3–5 seconds, phone held still): a table with an object, and you doing something a
character should react to: pointing, offering something, tapping the object. Leave clear space
where the character will be.

The free GPU only fits the **small** model (1.3B), so this shows the low end of open-source quality.
A run takes roughly 10–25 minutes.

In [ ]:
# 1. Install (about 1 minute)
!pip install -q "momentsgen[model,cutout] @ git+https://github.com/sasidhar8785/moments#subdirectory=server"
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload your clip

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
clip_path = Path(next(iter(uploaded)))
print("Clip:", clip_path)

## 3. Upload the character image
A clear, full-body picture of the character. The background is removed automatically.

In [ ]:
from PIL import Image
from momentsgen import prepare_reference
uploaded = files.upload()
reference = prepare_reference(Image.open(next(iter(uploaded))))
preview = reference.copy(); preview.thumbnail((300, 300)); display(preview)

## 4. Where the character goes, and what happens

Set the box (fractions of the frame: 0 = left/top edge, 1 = right/bottom edge) and describe the scene.
Make the box cover **everywhere the character will move**, not just where they start.
Re-run this cell to check the red box on your first frame.

In [ ]:
left = 0.35  #@param {type:"number"}
top = 0.3  #@param {type:"number"}
width = 0.4  #@param {type:"number"}
height = 0.6  #@param {type:"number"}
prompt = "A small grumpy man in a brown tweed suit stands on the table next to the toy car. He crosses his arms and complains, moving naturally throughout the shot. Realistic phone video, lit by the room's light."  #@param {type:"string"}
seconds = 3  #@param {type:"slider", min:1, max:5, step:1}

from PIL import ImageDraw
from momentsgen import Placement
from momentsgen import clip as clipio
placement = Placement(left, top, width, height)
first = Image.fromarray(clipio.read_frames(clip_path, 16, max_frames=1)[0])
draw = ImageDraw.Draw(first)
draw.rectangle(placement.pixels(*first.size), outline=(255, 0, 0), width=max(3, first.size[0] // 150))
first.thumbnail((480, 480)); display(first)

## 5. Load the model
Downloads about 19 GB the first time (a few minutes).

In [ ]:
mode = "fast"  #@param ["fast", "best"]
# fast: 4 steps. best: the full model, about 12x more work.
from momentsgen import BEST, FAST, load_pipeline
MODE = FAST if mode == "fast" else BEST
pipe = load_pipeline(mode=MODE)

## 6. Generate

In [ ]:
from momentsgen import insert_character
out_path = Path("moment.mp4")
try:
    insert_character(pipe, clip_path, reference, placement, prompt, out_path, max_seconds=seconds,
                     steps=MODE.steps, guidance=MODE.guidance)
except FloatingPointError:
    # Half precision overflowed on this GPU: reload in full precision (slower) and try again.
    del pipe; torch.cuda.empty_cache()
    pipe = load_pipeline(dtype=torch.float32, mode=MODE)
    insert_character(pipe, clip_path, reference, placement, prompt, out_path, max_seconds=seconds,
                     steps=MODE.steps, guidance=MODE.guidance)
print("Done:", out_path)

## 7. Watch and download

In [ ]:
from base64 import b64encode
from IPython.display import HTML
data = b64encode(out_path.read_bytes()).decode()
display(HTML(f'<video src="data:video/mp4;base64,{data}" controls playsinline style="max-height:600px"></video>'))
files.download(str(out_path))

Try again with a different box, prompt or seed by re-running steps 4 and 6 (the model stays loaded).
Send the result back so the settings can be tuned.